# WRDS Stock Risk and Return Analysis Workflow

This notebook documents the Python workflow behind the Streamlit app. It requires a valid WRDS account. Credentials are entered during runtime and are not saved in the notebook.

## 1. Problem definition

The analytical goal is to help an introductory finance user evaluate the historical return, risk, Sharpe Ratio, and drawdown profile of one listed stock using WRDS CRSP Flat File Format 2.0 (CIZ) daily stock data.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import getpass
import pandas as pd

from src.data_loader import connect_to_wrds, fetch_daily_stock_data, get_matching_securities
from src.metrics import calculate_performance_metrics, calculate_rolling_metrics, prepare_stock_data


## 2. Runtime inputs

The same inputs are exposed in the Streamlit app. The password input is hidden and should not be printed or saved.

In [ ]:
ticker = "AAPL"
start_date = "2023-01-01"
end_date = "2025-12-31"
annual_risk_free_rate = 0.02
rolling_window = 63


## 3. Connect to WRDS

Run this cell only when you are ready to connect to WRDS.

In [ ]:
wrds_username = input("WRDS username: ")
wrds_password = getpass.getpass("WRDS password: ")
conn = connect_to_wrds(wrds_username, wrds_password)


## 4. Match ticker to CRSP security

CRSP identifies securities using PERMNO. A ticker may map to more than one PERMNO over time, so the workflow first checks matching CRSP CIZ daily records.

In [ ]:
matches = get_matching_securities(conn, ticker, start_date, end_date)
matches.head()


In [ ]:
selected_security = matches.iloc[0]
selected_permno = int(selected_security["permno"])
selected_permno


## 5. Retrieve and prepare daily data

In [ ]:
raw_data = fetch_daily_stock_data(conn, selected_permno, start_date, end_date)
raw_data.head()


In [ ]:
stock_data = prepare_stock_data(raw_data)
stock_data[["date", "price", "daily_return", "cumulative_return", "drawdown"]].head()


## 6. Calculate performance metrics

In [ ]:
metrics = calculate_performance_metrics(stock_data, annual_risk_free_rate)
pd.Series(metrics)


In [ ]:
rolling_data = calculate_rolling_metrics(stock_data, rolling_window, annual_risk_free_rate)
rolling_data.tail()


## 7. Interpretation notes

- Total return shows the full holding-period performance.
- Annualized volatility measures historical return instability.
- Sharpe Ratio measures return earned per unit of risk after considering the risk-free rate.
- Maximum drawdown highlights the largest loss from a previous wealth peak.
- Rolling metrics show whether risk and risk-adjusted return were stable over time.